# Bonus — Combined Bid-Likelihood Score

**Goal:** Stack the two strongest signals from MVP 0 (Q1 adjacency + Q3 well activity) into a
single per-block score and evaluate its predictive power.

**Method:**
1. Compute block-level features: number of companies with adjacent leases, well count within 25 km / 6 months.
2. Fit a logistic regression (did_bid ~ features).
3. Evaluate with ROC-AUC and precision at top-N.

**Success:** AUC > 0.70 — the combined score meaningfully separates bid from no-bid blocks.

---
*Prototype using Sale 247 (March 2017).*

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from libpysal.weights import Rook
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
from sklearn.preprocessing import StandardScaler

# Colab: ROOT = repo clone location; local: two levels up from notebooks/
try:
    import google.colab  # noqa: F401
    ROOT = "/content/block-party"
except ImportError:
    ROOT = os.path.abspath(os.path.join("..", ".."))

sys.path.insert(0, os.path.join(ROOT, "mvp0"))
import boem_loader as bl

SALE_DATE = pd.Timestamp("2017-03-22")
WELL_RADIUS_KM = 25
WELL_WINDOW_MO = 6
print(f"Sale date: {SALE_DATE.date()}")
print(f"Well window: {WELL_WINDOW_MO} months, radius: {WELL_RADIUS_KM} km")

## 1. Load data

In [ ]:
sales  = bl.load_master_sales(os.path.join(ROOT, "data/lease-sales/master_lease_sales.csv"))
lh     = bl.load_lease_history(os.path.join(ROOT, "data/lease-sales/cleaned_lease_history.csv"))
lo     = bl.load_lease_owners(os.path.join(ROOT, "data/lease-sales/lseowndelimit.txt"))
blocks = bl.load_blocks(os.path.join(ROOT, "data/shapefiles/blocks.shp"), to_utm=True)
wells  = bl.load_boreholes(os.path.join(ROOT, "data/wells/mv_boreholes_all.txt"), region="G", to_utm=True)

print(f"Sale bids: {len(sales)} | Lease history: {len(lh)} | Owners: {len(lo)}")
print(f"Blocks: {len(blocks)} | Wells: {len(wells)}")

## 2. Build sale universe

In [ ]:
sale_prots = set(sales["Protraction_ID"].unique())
universe = blocks[blocks["Protraction_ID"].isin(sale_prots)].copy().reset_index(drop=True)
universe["block_idx"] = range(len(universe))

bid_pairs = set(zip(sales["Protraction_ID"], sales["Block_Number"]))
universe["did_bid"] = [
    (r["Protraction_ID"], r["Block_Number"]) in bid_pairs
    for _, r in universe.iterrows()
]
print(f"Universe: {len(universe)} blocks, {universe['did_bid'].sum()} with bids")

## 3. Feature 1 — Adjacency interest count

For each block, count how many distinct companies hold an active lease on an
edge-adjacent block.  More companies nearby → more competitive interest → higher
bid likelihood.

In [ ]:
# Use the canonical helper from boem_loader (avoids the TERMED/TERMIN typo and code duplication)
active_co = bl.active_leases_at(lh, lo, SALE_DATE)

# Rename clashing columns produced by the inner merge
if "Area_Code_lh" in active_co.columns:
    active_co = active_co.rename(columns={"Area_Code_lh": "Area_Code"})

active_blocks = active_co.merge(
    universe[["AREA_CODE", "Block_Number", "block_idx"]],
    left_on=["Area_Code", "Block_Number"],
    right_on=["AREA_CODE", "Block_Number"],
    how="inner",
)
print(f"Active leases in universe: {len(active_blocks)}")
print(f"Unique companies: {active_blocks['Company_Number'].nunique()}")

In [ ]:
# Rook adjacency
w = Rook.from_dataframe(universe, use_index=False)
adj = {i: set(w.neighbors[i]) for i in range(w.n)}

# For each block, count distinct companies on adjacent blocks
# Build: block_idx → set of Company_Numbers holding leases
block_companies = (
    active_blocks.groupby("block_idx")["Company_Number"]
    .apply(set)
    .to_dict()
)

adj_company_counts = []
for idx in range(len(universe)):
    neighbor_cos = set()
    for ni in adj.get(idx, set()):
        neighbor_cos.update(block_companies.get(ni, set()))
    adj_company_counts.append(len(neighbor_cos))

universe["n_adj_companies"] = adj_company_counts

print(f"Blocks with >= 1 adjacent company: {(universe['n_adj_companies'] > 0).sum()}")
print(universe["n_adj_companies"].describe().to_string())

## 4. Feature 2 — Well activity count

Count wells spudded within 25 km in the 6 months before the sale (same as the
best-performing Q3 configuration).

In [ ]:
well_start = SALE_DATE - pd.DateOffset(months=WELL_WINDOW_MO)
recent_wells = wells[
    (wells["Spud_Date"] >= well_start) & (wells["Spud_Date"] <= SALE_DATE)
].copy()
print(f"Wells in {WELL_WINDOW_MO}-month window: {len(recent_wells)}")

# Buffer wells and spatial join to block centroids
centroids = gpd.GeoDataFrame(
    universe[["block_idx"]],
    geometry=universe.geometry.centroid,
    crs=universe.crs,
)

buffered = recent_wells.copy()
buffered["geometry"] = buffered.geometry.buffer(WELL_RADIUS_KM * 1000)

joined = gpd.sjoin(centroids, buffered[["geometry"]], how="left", predicate="within")
well_count = (
    joined.groupby(joined.index)["index_right"]
    .count()
    .reindex(range(len(universe)), fill_value=0)
)
universe["n_wells_nearby"] = well_count.values

print(f"Blocks with >= 1 well nearby: {(universe['n_wells_nearby'] > 0).sum()}")
print(universe["n_wells_nearby"].describe().to_string())

## 5. Feature overview

In [ ]:
features = ["n_adj_companies", "n_wells_nearby"]
X = universe[features].values
y = universe["did_bid"].astype(int).values

print(f"Feature matrix: {X.shape[0]} blocks × {X.shape[1]} features")
print(f"Positive class (bid): {y.sum()} ({y.mean():.2%})")
print(f"Negative class (no bid): {(1 - y).sum()}")

# Correlation between features
corr = np.corrcoef(X[:, 0], X[:, 1])[0, 1]
print(f"\nPearson correlation between features: {corr:.3f}")

# Per-feature bid rate lift
for feat in features:
    has = universe[universe[feat] > 0]["did_bid"].mean()
    has_not = universe[universe[feat] == 0]["did_bid"].mean()
    lift = has / has_not if has_not > 0 else float("inf")
    print(f"{feat}: bid rate {has:.2%} (present) vs {has_not:.2%} (absent) → {lift:.1f}x lift")

## 6. Logistic regression with cross-validation

**Class imbalance note:** Only ~2 % of blocks receive bids.  `class_weight='balanced'`
re-weights the loss function so the minority class (bid=1) is not swamped.

**Evaluation note:** The in-sample AUC re-uses training data and is optimistically
biased.  5-fold stratified CV gives a more honest estimate.  The true test is a
held-out sale (Sale 257, 261, or Dec 2025).

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

rng = np.random.default_rng(42)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# class_weight='balanced' compensates for the ~50:1 class imbalance
model = LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced")
model.fit(X_scaled, y)

# In-sample score (optimistic upper bound)
universe["score"] = model.predict_proba(X_scaled)[:, 1]

# Cross-validated metrics (more honest)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_aucs = cross_val_score(model, X_scaled, y, cv=cv, scoring="roc_auc")
cv_aps  = cross_val_score(model, X_scaled, y, cv=cv, scoring="average_precision")

print("Logistic regression coefficients (standardized):")
for feat, coef in zip(features, model.coef_[0]):
    print(f"  {feat:20s}: {coef:+.4f}")
print(f"  {'intercept':20s}: {model.intercept_[0]:+.4f}")
print(f"\nClass balance: {y.sum()} bids / {len(y)} blocks = {y.mean():.2%}")
print(f"\n=== In-sample (optimistic) ===")
print(f"  ROC-AUC:       {roc_auc_score(y, universe['score']):.4f}")
print(f"  Avg Precision: {average_precision_score(y, universe['score']):.4f}  (baseline: {y.mean():.4f})")
print(f"\n=== 5-fold CV (more honest) ===")
print(f"  ROC-AUC:       {cv_aucs.mean():.4f} ± {cv_aucs.std():.4f}")
print(f"  Avg Precision: {cv_aps.mean():.4f} ± {cv_aps.std():.4f}")

## 7. ROC-AUC evaluation

In [ ]:
auc = roc_auc_score(y, universe["score"])
ap = average_precision_score(y, universe["score"])
print(f"ROC-AUC:          {auc:.4f}")
print(f"Avg Precision:    {ap:.4f}")

# Individual feature AUCs for comparison
for i, feat in enumerate(features):
    feat_auc = roc_auc_score(y, X[:, i])
    print(f"AUC ({feat} alone): {feat_auc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC curve
ax = axes[0]
fpr, tpr, _ = roc_curve(y, universe["score"])
ax.plot(fpr, tpr, "b-", linewidth=2, label=f"Combined (AUC={auc:.3f})")
for i, feat in enumerate(features):
    feat_fpr, feat_tpr, _ = roc_curve(y, X[:, i])
    feat_auc = roc_auc_score(y, X[:, i])
    ax.plot(feat_fpr, feat_tpr, "--", linewidth=1, label=f"{feat} (AUC={feat_auc:.3f})")
ax.plot([0, 1], [0, 1], "k:", alpha=0.4)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Precision-Recall curve
ax = axes[1]
prec, rec, _ = precision_recall_curve(y, universe["score"])
ax.plot(rec, prec, "b-", linewidth=2, label=f"Combined (AP={ap:.3f})")
for i, feat in enumerate(features):
    feat_prec, feat_rec, _ = precision_recall_curve(y, X[:, i])
    feat_ap = average_precision_score(y, X[:, i])
    ax.plot(feat_rec, feat_prec, "--", linewidth=1, label=f"{feat} (AP={feat_ap:.3f})")
baseline = y.mean()
ax.axhline(baseline, color="k", linestyle=":", alpha=0.4, label=f"Baseline ({baseline:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle("Combined Score: Q1 Adjacency + Q3 Well Activity", fontsize=13)
plt.tight_layout()
plt.show()

## 8. Precision at top-N

If we rank all blocks by score and flag the top N, how many actual bids do we capture?

In [ ]:
ranked = universe.sort_values("score", ascending=False).reset_index(drop=True)
ranked["rank"] = range(1, len(ranked) + 1)
ranked["cum_bids"] = ranked["did_bid"].cumsum()

total_bids = y.sum()

top_ns = [50, 100, 200, 300, 500, 1000]
print(f"{'Top-N':>8}  {'Precision':>10}  {'Recall':>8}  {'Bids found':>11}")
print("-" * 45)
for n in top_ns:
    if n > len(ranked):
        continue
    top = ranked.head(n)
    prec_n = top["did_bid"].mean()
    recall_n = top["did_bid"].sum() / total_bids
    print(f"{n:>8}  {prec_n:>10.1%}  {recall_n:>8.1%}  {int(top['did_bid'].sum()):>6} / {total_bids}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ns = np.arange(1, len(ranked) + 1)
cum_recall = ranked["cum_bids"].values / total_bids
cum_precision = ranked["cum_bids"].values / ns

ax.plot(ns, cum_recall * 100, "b-", linewidth=2, label="Recall (% bids captured)")
ax.plot(ns, cum_precision * 100, "r-", linewidth=2, label="Precision (% flagged that are bids)")
ax.axhline(baseline * 100, color="gray", linestyle=":", label=f"Random precision ({baseline:.1%})")

ax.set_xlabel("Blocks flagged (top-N by score)")
ax.set_ylabel("%")
ax.set_title("Precision & Recall vs. number of blocks flagged")
ax.set_xlim(0, 2000)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Score distribution and feature contribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Score distribution by class
ax = axes[0]
ax.hist(universe.loc[universe["did_bid"], "score"], bins=40, alpha=0.7,
        color="#2196F3", label="Bid", density=True)
ax.hist(universe.loc[~universe["did_bid"], "score"], bins=40, alpha=0.5,
        color="#9E9E9E", label="No bid", density=True)
ax.set_xlabel("Predicted score")
ax.set_ylabel("Density")
ax.set_title("Score distribution")
ax.legend()

# Scatter: feature 1 vs feature 2, colored by bid
ax = axes[1]
no_bid = universe[~universe["did_bid"]]
bid = universe[universe["did_bid"]]
ax.scatter(no_bid["n_adj_companies"], no_bid["n_wells_nearby"],
           c="#BDBDBD", s=5, alpha=0.3, label="No bid")
ax.scatter(bid["n_adj_companies"], bid["n_wells_nearby"],
           c="#F44336", s=30, edgecolors="black", linewidth=0.5, label="Bid", zorder=5)
ax.set_xlabel("Adjacent companies")
ax.set_ylabel("Wells nearby (25km/6mo)")
ax.set_title("Feature space")
ax.legend(fontsize=8)

# Coefficient bar chart
ax = axes[2]
coefs = pd.Series(model.coef_[0], index=features)
coefs.plot.barh(ax=ax, color=["#2196F3", "#4CAF50"], edgecolor="black")
ax.set_xlabel("Logistic regression coefficient (standardized)")
ax.set_title("Feature importance")

plt.suptitle("Combined Score Analysis", fontsize=13)
plt.tight_layout()
plt.show()

## 10. Map — top-scored blocks vs. actual bids

In [ ]:
# Flag top-200 blocks as predictions
TOP_N = 200
top_idxs = set(ranked.head(TOP_N).index)
universe["top_predicted"] = universe.index.isin(top_idxs)

from matplotlib.patches import Patch

fig, ax = plt.subplots(figsize=(14, 9))
universe.plot(ax=ax, color="whitesmoke", edgecolor="gray", linewidth=0.1)

# Predicted blocks
predicted = universe[universe["top_predicted"]]
predicted.plot(ax=ax, color="#BBDEFB", edgecolor="#1565C0", linewidth=0.4)

# True positive = predicted + bid
tp = universe[universe["top_predicted"] & universe["did_bid"]]
tp.plot(ax=ax, color="#2196F3", edgecolor="black", linewidth=0.8)

# False negative = bid but not predicted
fn = universe[~universe["top_predicted"] & universe["did_bid"]]
fn.plot(ax=ax, color="#F44336", edgecolor="black", linewidth=0.8)

prec_top = tp.shape[0] / TOP_N if TOP_N > 0 else 0
recall_top = tp.shape[0] / total_bids if total_bids > 0 else 0

ax.set_title(f"Top-{TOP_N} Prediction Map — Precision {prec_top:.0%}, Recall {recall_top:.0%}")

# Create manual legend to avoid PatchCollection warning
legend_elements = [
    Patch(facecolor="#BBDEFB", edgecolor="#1565C0", label=f"Top {TOP_N} predicted"),
    Patch(facecolor="#2196F3", edgecolor="black", label="True positive (predicted + bid)"),
    Patch(facecolor="#F44336", edgecolor="black", label="Missed bid (not in top-N)"),
]
ax.legend(handles=legend_elements, loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

## 11. Interpretation

| AUC | Interpretation | MVP 1 action |
|-----|---------------|-------------|
| > 0.80 | Strong — the two signals jointly identify bid targets | Ship as core ranking in MVP 1 |
| 0.70–0.80 | Good — meaningful separation, room to improve | Ship as primary ranking, plan to add features |
| 0.60–0.70 | Marginal — better than random, but noisy | Use as one input alongside expert judgment |
| < 0.60 | Weak — not practically useful alone | Re-evaluate which signals to combine |

**Note:** This is an in-sample evaluation on Sale 247. True out-of-sample performance
requires testing on a held-out sale (e.g., Dec 2025 when available).